# **Задание №6. Разработка спецификации требований к информационной системе**


## 0. Область и цели

Цель системы — предоставить веб-платформу, в которой авторизованный пользователь создаёт и редактирует квизы, автоматически генерируя вопросы **множественного выбора** с помощью локально развернутой LLM (Llama 3.1), публикует квизы и собирает результаты прохождений. Система работает **без внешних LLM-облаков**.

**Границы релиза v1:** генерация и редактирование вопросов множественного выбора, публикация, прохождение, результаты, базовая аналитика, управление пользователями (админ), аудит и резервное копирование.

**Ключевое допущение об источнике контента:** авторизованный пользователь задаёт **тему** и при желании вставляет **контекст (вставка текста)** в специальное поле (до X символов, см. FR-003). Загрузка файлов/URL не входит в v1 (может появиться в будущих релизах).

## 1. Роли и доступ

**R1. Администратор (ADMIN):** управление пользователями/правами, системные настройки, бэкапы/восстановление, просмотр аудита.

**R2. Авторизованный пользователь (USER):** создание/редактирование квизов, запуск генерации вопросов, публикация, прохождение квизов (своих и доступных), просмотр результатов.

## 2. Функциональные требования (FR)

*Формат: ID — Формулировка — Приоритет — Критерии проверки*

**FR-001 Аутентификация по паролю** — Система должна поддерживать вход/выход по email+паролю с политикой сложности паролей (≥8 символов, 1 цифра, 1 буква).
Приоритет: Критический.
Проверка: создать пользователя, вход с корректными/некорректными данными; попытка входа с простым паролем отклоняется; 10 неуспешных попыток → временная блокировка 15 минут.

**FR-002 Роли и авторизация** — Роли ADMIN и USER, доступ к экранам и API по RBAC.
Приоритет: Критический.
Проверка: под учеткой USER доступ к разделам администрирования запрещён (HTTP 403); под ADMIN — разрешён.

**FR-003 Создание черновика квиза** — Пользователь может создать квиз, указав: *Название (1–120 симв)*, *Тему*, *Опциональный контекст* (до **50 000** символов), *Желаемое число вопросов* (1–50) и *уровень сложности* (низкий/средний/высокий).
Приоритет: Высокий.
Проверка: при валидных данных создаётся черновик в БД; при превышении лимита контекста — валидационная ошибка.

**FR-004 Генерация вопросов (MCQ) локальной LLM** — По кнопке «Сгенерировать» система запускает локальный инференс Llama 3.1 и возвращает не менее N вопросов (N задано в FR-003) формата: *текст вопроса*, **4 варианта** (A–D) с ровно **1 правильным**, *объяснение ответа* (опционально, если задано в настройках генерации).
Приоритет: Критический.
Проверка: при генерации 10 вопросов возвращается список длиной 10; каждый элемент содержит 4 опции и 1 помечен правильным; формат валидируется схемой JSON.

**FR-005 Настройки генерации** — Пользователь может указать параметры: *стиль (тест/экзамен/практика)*, *макс. длина вопроса*, *требовать объяснения*, *уровень сложности*.
Приоритет: Высокий.
Проверка: изменение настроек приводит к изменению промпта; при включенном «требовать объяснения» каждый вопрос содержит поле explanation ≠ пусто.

**FR-006 Редактор вопросов** — Пользователь может редактировать сгенерированный вопрос: текст, варианты, правильный ответ, теги; помечать «требует правки».
Приоритет: Критический.
Проверка: CRUD-операции над вопросом сохраняются; аудит фиксирует изменения (см. FR-013).

**FR-007 Статусы квиза** — *Черновик → Готов к публикации → Опубликован → Архив*.
Приоритет: Высокий.
Проверка: недопустимы пропуски стадий (напр., из Черновика в Архив); при публикации фиксируется timestamp; опубликованный квиз доступен для прохождения по ссылке (для авторизованных).

**FR-008 Прохождение квиза** — Авторизованный пользователь может пройти опубликованный квиз: последовательная/свободная навигация, таймер (опционально), одна или несколько попыток (по настройке квиза).
Приоритет: Критический.
Проверка: старт попытки создаёт запись; отправка фиксирует ответы и вычисляет балл; при истечении таймера попытка авто-завершается.

**FR-009 Подсчёт результатов и отчёт** — Система считает итоговый балл (% правильных), показывает список вопросов с выбранными/правильными и (если есть) объяснениями; сохраняет в БД.
Приоритет: Критический.
Проверка: на тестовом квизе с известным ключом результат совпадает с ожидаемым (100% при всех правильных).

**FR-010 Аналитика по квизу** — Автор видит агрегаты: число попыток, средний балл, распределение по вопросам (доля правильных), среднее время прохождения.
Приоритет: Средний.
Проверка: дашборд отображает корректные метрики на данных-фикстурах.

**FR-011 Поиск и фильтрация** — Поиск по названию/теме/тегам; фильтры по статусу, дате, автору (для ADMIN — по всем).
Приоритет: Средний.
Проверка: запрос «тема: Биология» возвращает только совпадения; пагинация работает.

**FR-012 Экспорт/импорт квиза** — Экспорт в JSON (структура вопросов и метаданных); импорт JSON с валидацией схемы.
Приоритет: Средний.
Проверка: экспортированный файл можно импортировать обратно без потерь; неправильная схема — понятная ошибка.

**FR-013 Аудит действий** — Фиксировать создание/редактирование/публикацию/удаление квизов и вопросы (кто, когда, что).
Приоритет: Высокий.
Проверка: в журнале появляется запись после каждой операции; чтение журнала доступно ADMIN.

**FR-014 Управление пользователями (ADMIN)** — Создание/блокировка/сброс пароля; назначение роли.
Приоритет: Высокий.
Проверка: заблокированная учётка не может войти (HTTP 401/403).

**FR-015 Резервное копирование и восстановление** — ADMIN может запускать on-demand бэкап БД и восстанавливать из бэкапа.
Приоритет: Высокий.
Проверка: после бэкапа создаётся архив; после восстановления данные соответствуют состоянию на момент бэкапа.

**FR-016 Настройки попыток** — Для квиза задаются: лимит попыток (1–10), тайм-лимит (0=без лимита или 1–180 минут), показывать объяснения после сдачи (да/нет).
Приоритет: Средний.
Проверка: при лимите=1 вторая попытка невозможна; таймер завершает попытку.

**FR-017 Публичная ссылка для авторизованных** — Генерация короткой ссылки/QR, ведущей на страницу логина и затем — на квиз (только для авторизованных).
Приоритет: Низкий.
Проверка: по ссылке без авторизации редирект на login; после логина — старт квиза.

**FR-018 Локализация интерфейса** — Русский язык интерфейса по умолчанию; все пользовательские тексты доступны из словаря.
Приоритет: Средний.
Проверка: UI-строки не «зашиты» в код; 100% строк покрыто словарём.


## 3. Нефункциональные требования (NFR)


### 3.1 Производительность

* **NFR-P01**: P95 время открытия «Мои квизы» ≤ **1.2 с** при 200 одновременных сессиях. *Обоснование: типичные RPS и таблицы до 1000 записей.*
  Метод проверки: JMeter/Locust, сценарий навигации.
* **NFR-P02**: P95 старт прохождения квиза (загрузка вопросов) ≤ **2.0 с** (кешированные данные).
  Метод: нагрузочный тест с 200 конкурентами.
* **NFR-P03**: P95 генерация **10** вопросов LLM ≤ **30 с** на референс-хосте: **8 vCPU, 32 GB RAM, без GPU, Llama 3.1 8B Q4**. *Обоснование: CPU-инференс квантизованной модели.*
  Метод: 30 запусков подряд, измерение времени.
* **NFR-P04**: Падение производительности при 50 одновременных генерациях не превышает 2× против одиночной благодаря очереди задач.
  Метод: стресс-тест; мониторинг P95.

### 3.2 Надёжность и доступность

* **NFR-R01**: Доступность сервиса (App/API) — **≥99.5%** в месяц; плановые окна — до 2 ч/мес ночью.
  Метод: аптайм-мониторинг.
* **NFR-R02**: **RTO ≤ 4 ч**, **RPO ≤ 24 ч**.
  Метод: учения DR: восстановление из вчерашнего бэкапа.
* **NFR-R03**: Все операции сохранения — атомарны; при сбое во время отправки попытки ответы не теряются.
  Метод: интеграционные тесты с эмуляцией отказа.

### 3.3 Безопасность

* **NFR-S01**: TLS 1.3 для всего трафика; шифрование данных «в покое» (AES-256 для диска/бэкапов).
  Метод: проверка конфигураций.
* **NFR-S02**: Хеширование паролей — **Argon2id**; политика: минимум 8 символов; блокировка после 10 неудачных попыток на 15 мин.
  Метод: ревью и тесты аутентификации.
* **NFR-S03**: RBAC по ролям; запрет эскалации привилегий; аудит всех административных действий (сохранение ≥90 дней).
  Метод: пен-тест и просмотр журнала.
* **NFR-S04**: Защита от OWASP Top-10: CSRF-токены, проверка вводов, лимитирование запросов к логину (≤ 10/мин/аккаунт), HTTP security headers.
  Метод: SAST/DAST (например, ZAP).
* **NFR-S05**: Локальная LLM без передачи данных третьим лицам; возможность «офлайн» работы сервера.
  Метод: анализ архитектуры сети.

### 3.4 Удобство использования (UX)

* **NFR-U01**: Новому пользователю требуется **≤10 мин** на создание и публикацию первого квиза из 10 вопросов (видео-сценарий + тест наблюдения на 5 респондентах).
* **NFR-U02**: Начать прохождение опубликованного квиза — **≤3 кликов** от главной страницы.
* **NFR-U03**: Доступность **WCAG 2.1 AA** (фокус, контраст, навигация с клавиатуры).
  Метод: axe-сканер + экспертная проверка.

### 3.5 Совместимость и переносимость

* **NFR-C01 (Клиент)**: Поддержка последних двух версий Chrome, Edge, Firefox и **Safari 15+**; адаптивная вёрстка (≥360 px ширина).
  Метод: кросс-браузерные прогон-листы.
* **NFR-C02 (Сервер)**: Деплой в контейнере **Docker 24+**; базовая оркестрация Docker Compose; ОС — Linux (Ubuntu LTS). БД — **PostgreSQL 15**.
  Метод: установка на чистую VM по инструкции.
* **NFR-C03 (Переносимость)**: Возможность установки в изолированной сети (без доступа в интернет) с локальным реестром образов.
  Метод: dry-run в изолированном стенде.

### 3.6 Масштабируемость и наблюдаемость

* **NFR-SC01**: Горизонтальное масштабирование веб-приложения (статус-без-состояния); sticky-sessions не требуются.
* **NFR-SC02**: Воркеры инференса LLM масштабируются по числу процессов; очередь задач гарантирует FIFO.
* **NFR-O01**: Метрики и трассировка (OpenTelemetry/Prometheus), дашборд с P95 латентности, ошибками, глубиной очереди.
  Метод: показ дашборда в staging.

### 3.7 Поддерживаемость

* **NFR-M01**: Покрытие модульными тестами ≥ **70%** по строкам в core-модулях.
* **NFR-M02**: API версионируется (v1) и документируется (OpenAPI).
  Метод: отчёт CI.

## 4. Варианты использования (Use Cases)

**UC-01 Создание и публикация квиза** (Актор: USER)

1. USER открывает «Создать квиз», вводит *Название*, *Тему*, вставляет при необходимости *Контекст* (загружает файл), указывает *N=10*, *Сложность=средняя*.
2. Нажимает «Сгенерировать» → очередь задач → LLM создаёт 10 вопросов MCQ.
3. USER редактирует отдельные вопросы (правит формулировки/варианты/правильный ответ, добавляет теги).
4. Переводит статус «Готов к публикации» → «Опубликовать».
   **Результат:** квиз доступен по ссылке для авторизованных.
   **Критерий успеха:** на экране предпросмотра видно 10 валидных вопросов.

**UC-02 Прохождение квиза** (Актор: USER)

1. USER открывает ссылку квиза (после логина).
2. Запускает попытку; при активном таймере отображается обратный отсчёт.
3. Отвечает на вопросы; завершает попытку.
   **Результат:** система показывает % правильных, правильные ответы и объяснения (если включены).
   **Критерий:** запись результатов создана; балл рассчитан корректно.

**UC-03 Аналитика автора** (Актор: USER)
Просмотр агрегатов: число попыток, средний балл, сложность вопросов (по тегам), проблемные вопросы (низкая доля правильных).
**Критерий:** графики соответствуют данным.

**UC-04 Управление пользователями** (Актор: ADMIN)
Создание учетной записи, блокировка, сброс пароля.
**Критерий:** заблокированный пользователь не входит.

**UC-05 Бэкап/восстановление** (Актор: ADMIN)
Создание бэкапа и восстановление из архива.
**Критерий:** после восстановления данные соответствуют точке RPO.


## 5. Ограничения и допущения

### 5.1 Технологические ограничения

* Локальная LLM **Llama 3.1 (8B, Q4)**, CPU-инференс; отсутствие внешних LLM-API.
* Сервер Linux + Docker; БД PostgreSQL 15.
* Очередь задач (например, Redis/RQ) для генераций.

### 5.2 Бизнес-ограничения

* Поддерживаемый тип вопросов в v1 — **только множественный выбор (1 правильный из 4)**.
* Публичный доступ без авторизации **не допускается** (все прохождения — только авторизованные).

### 5.3 Допущения среды эксплуатации

* Развёртывание на одной VM (8 vCPU, 32 GB RAM, SSD) с возможностью горизонтального масштабирования приложения и воркеров.
* Сеть может быть изолированной (нет выхода в интернет).
